In [17]:
import cv2
import numpy as np
import time

In [18]:
CAMERA_INDEX      = 0      
WIDTH, HEIGHT     = 640, 480
BG_FRAME_COUNT    = 120     
WAIT_SECONDS      = 5       
DIFF_THRESHOLD    = 18      
MIN_PERSON_AREA   = 5000 

In [ ]:
def capture_background(cap):
    """Step out of frame — this captures the clean background."""
    print("\n  MOVE OUT of the frame completely")
    for i in range(WAIT_SECONDS, 0, -1):
        print(f"   {i}...")
        time.sleep(1)
    print("Capturing background please stay out")
    frames = []
    for _ in range(BG_FRAME_COUNT):
        ret, frame = cap.read()
        if ret:
            frames.append(frame.astype(np.float32))
    background = np.median(frames, axis=0).astype(np.uint8)
    print(" Background captured You can come back now.\n")
    return background

In [20]:
def get_person_mask(frame, bg_gray):
    """Detects where the person is in the frame using background subtraction."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32)
    diff = cv2.absdiff(gray, bg_gray)
    _, mask = cv2.threshold(diff, DIFF_THRESHOLD, 255, cv2.THRESH_BINARY)
    mask = mask.astype(np.uint8)
    small = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, small, iterations=1)
    large = np.ones((15, 15), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, large, iterations=6)
    mask = cv2.dilate(mask, large, iterations=2)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clean = np.zeros_like(mask)
    for cnt in contours:
        if cv2.contourArea(cnt) > MIN_PERSON_AREA:
            cv2.drawContours(clean, [cnt], -1, 255, thickness=cv2.FILLED)
    clean = cv2.GaussianBlur(clean, (31, 31), 0)
    return clean

In [21]:
def apply_invisibility(frame, background, mask, alpha):
    """Blends background into the person region to create invisibility."""
    blend = (mask.astype(np.float32) / 255.0) * alpha
    blend_3ch = np.stack([blend] * 3, axis=-1)
    frame_f = frame.astype(np.float32)
    bg_f    = background.astype(np.float32)
    output = (bg_f * blend_3ch + frame_f * (1.0 - blend_3ch))
    return output.astype(np.uint8)

In [22]:
def draw_ui(frame, alpha):
    """Draws the on-screen controls and status."""
    percent = int(alpha * 100)
    cv2.rectangle(frame, (0, 0), (640, 75), (0, 0, 0), -1)
    bar_x = int(4.5 * percent)
    cv2.rectangle(frame, (10, 45), (10 + bar_x, 65), (0, 255, 255), -1)
    cv2.rectangle(frame, (10, 45), (460, 65), (255, 255, 255), 1)
    cv2.putText(frame, f"Invisibility: {percent}%",
                (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(frame, "+ Increase   - Decrease   Q Quit   S Screenshot",
                (10, 460), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    return frame

In [23]:
def main():
    cap = cv2.VideoCapture(CAMERA_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)
    print("   AI INVISIBILITY CLOAK")
    background = capture_background(cap)
    bg_gray    = cv2.cvtColor(background, cv2.COLOR_BGR2GRAY).astype(np.float32)
    alpha = 1.0          
    screenshot_count = 0
    print("Controls:")
    print("  +   →  Increase invisibility")
    print("  -   →  Decrease invisibility")
    print("  S   →  Save screenshot")
    print("  Q   →  Quit\n")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)   
        mask   = get_person_mask(frame, bg_gray)
        output = apply_invisibility(frame, background, mask, alpha)
        output = draw_ui(output, alpha)
        cv2.imshow("AI Invisibility ", output)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key in (ord('+'), ord('=')):
            alpha = min(1.0, round(alpha + 0.1, 1))
        elif key == ord('-'):
            alpha = max(0.0, round(alpha - 0.1, 1))
        elif key == ord('s'):
            filename = f"invisible{screenshot_count}.png"
            cv2.imwrite(filename, output)
            print(f" Screenshot saved → {filename}")
            screenshot_count += 1
    cap.release()
    cv2.destroyAllWindows()

In [24]:
if __name__ == "__main__":
    main()

   AI INVISIBILITY CLOAK

  MOVE OUT of the frame completely
   5...
   4...
   3...
   2...
   1...
Capturing background please stay out
 Background captured! You can come back now.

Controls:
  +   →  Increase invisibility
  -   →  Decrease invisibility
  S   →  Save screenshot
  Q   →  Quit

 Screenshot saved → invisible0.png
